# DispatchDesk Tool-Calling Eval

Measures how reliably the agent selects the right tool, passes correct arguments,
and grounds its answer in the tool result.

The LLM runs for real (each case makes one or more Groq calls) — tool-selection
behaviour is tested against the actual model. Tool *execution* is mocked with
deterministic canned responses so the eval is repeatable without a live database.

**Five metrics per case:**

| Metric | Definition |
|---|---|
| Tool selection accuracy | Called tools exactly match expected tools |
| No spurious calls | Agent never calls a tool that wasn't needed |
| No missing calls | Agent never skips a required tool |
| Argument accuracy | Fraction of expected args passed correctly (0–1) |
| Answer groundedness | Final answer contains all expected key facts from the mock response |

**25 eval cases** covering live-status, historical metrics, policy (no-tool),
multi-tool comparisons, and adversarial edge cases.

Cases and results live in `evals/tool_calling/`. Run this notebook from `backend/`
with `uv run jupyter notebook`.

## Setup

Adds the backend and eval directories to `sys.path` and imports the runner module.
The runner handles backend bootstrapping, Groq 429 retry logic, mocked tool dispatch,
scoring, and result persistence — the notebook stays thin.

In [1]:
import json
import sys
from pathlib import Path


def _find_dirs() -> tuple[Path, Path]:
    current = Path.cwd().resolve()
    for directory in (current, *current.parents):
        for candidate in (directory, directory / "backend"):
            if (candidate / "service" / "agent.py").is_file():
                project_root = candidate.parent
                return candidate, project_root / "evals" / "tool_calling"
    raise FileNotFoundError("Run from within the DispatchDesk repository")


BACKEND_DIR, EVAL_DIR = _find_dirs()
for p in (str(BACKEND_DIR), str(EVAL_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import run_eval

print(f"Backend : {BACKEND_DIR}")
print(f"Eval dir: {EVAL_DIR}")
print(f"Cases   : {run_eval.CASES_FILE}  ({len(json.loads(run_eval.CASES_FILE.read_text()))} cases)")

Backend : /Users/ravi/Projects/supply-chain-bv/backend
Eval dir: /Users/ravi/Projects/supply-chain-bv/evals/tool_calling
Cases   : /Users/ravi/Projects/supply-chain-bv/evals/tool_calling/cases.json  (25 cases)


## Eval dataset

Cases are defined in `evals/tool_calling/cases.json`. Each case specifies:
- `question` / `store_id` — inputs to the agent
- `expected_tools` — tool names that should (and only should) be called
- `expected_args` — required argument values keyed by tool name
- `answer_keywords` — substrings that must appear in the final answer (groundedness check)

In [2]:
cases = json.loads(run_eval.CASES_FILE.read_text())
print(f"{len(cases)} eval cases:\n")
for c in cases:
    tools = c['expected_tools'] or ['(none)']
    kws   = c.get('answer_keywords', [])
    print(f"  {c['id']:14}  {c['label']:45}  tools={tools}  keywords={kws}")

25 eval cases:

  live-01         Available rider count                          tools=['get_live_dispatch_status']  keywords=['4']
  live-02         Packed-waiting orders                          tools=['get_live_dispatch_status']  keywords=['ORD-01-001', 'ORD-01-002']
  live-03         Rider break check                              tools=['get_live_dispatch_status']  keywords=['95']
  live-04         Riders returning within 10 min                 tools=['get_live_dispatch_status']  keywords=['Lakshmi']
  live-05         Rider zone lookup                              tools=['get_live_dispatch_status']  keywords=['Z-A']
  live-06         Long-shift riders                              tools=['get_live_dispatch_status']  keywords=['5.2']
  live-07         Standby off-shift riders                       tools=['get_live_dispatch_status']  keywords=['Kiran']
  metrics-01      SLA on 2026-10-07 20-22                        tools=['get_delivery_metrics']  keywords=['37']
  metrics-02      Ord

## Run eval

Each case makes 1–3 real Groq API calls. Tool execution is mocked.

**Rate limiting:** Groq free tier caps at 8 000 tokens/min and has an hourly
budget. The runner waits 65 s between cases and uses per-call retry with
exponential back-off on 429s (respects `retry-after` header). If a long wait
appears in the output that is normal — just let it run.

Total wall time: ~30 min for 25 cases (mostly inter-case sleep).

In [ ]:
scores, aggregate, run_ts = run_eval.run(verbose=True)

Running 25 eval cases (real Groq API calls, mocked tool execution) ...
  [ 1/25] live-01       2026-10-09 18:03:12 [info     ] Tool call                      step=1 tool=get_live_dispatch_status
2026-10-09 18:03:13 [info     ] Agent finished                 llm_calls=2 store_id=DS-BLR-014 tool_calls=1
tools=['get_live_dispatch_status']  llm_calls=2  1.57s
  [ 2/25] live-02       2026-10-09 18:04:19 [info     ] Tool call                      step=1 tool=get_live_dispatch_status
2026-10-09 18:04:20 [info     ] Agent finished                 llm_calls=2 store_id=DS-BLR-014 tool_calls=1
tools=['get_live_dispatch_status']  llm_calls=2  1.53s
  [ 3/25] live-03       
    [call-1 429 — waiting 305s] 
    [call-2 429 — waiting 65s] 2026-10-09 18:11:36 [info     ] Tool call                      step=1 tool=get_live_dispatch_status

    [call-1 429 — waiting 1808s] 

## Results

Reads the JSON written by the runner for the current `run_ts`.

In [ ]:
result_file = run_eval.RESULTS_DIR / f"results_{run_ts}.json"
payload = json.loads(result_file.read_text())

print(f"Run: {payload['run_ts']}  |  {payload['aggregate']['n_cases']} cases\n")
a = payload["aggregate"]
print(f"  Tool selection accuracy  {a['tool_selection_accuracy']:.0%}")
print(f"  No spurious calls        {a['no_spurious_calls_rate']:.0%}")
print(f"  No missing calls         {a['no_missing_calls_rate']:.0%}")
if a["argument_accuracy_mean"] is not None:
    print(f"  Argument accuracy (mean) {a['argument_accuracy_mean']:.0%}  (n={a['argument_accuracy_n']})")
if a["answer_groundedness_rate"] is not None:
    print(f"  Answer groundedness      {a['answer_groundedness_rate']:.0%}  (n={a['answer_groundedness_n']})")
print(f"  Avg LLM calls/query      {a['avg_llm_calls']:.1f}")
print(f"  Total wall time          {a['total_elapsed_sec']:.0f}s")

print("\nFiles written:")
for suffix in (".json", ".csv", "_trace.csv"):
    p = run_eval.RESULTS_DIR / f"results_{run_ts}{suffix}"
    rel = p.relative_to(run_eval.RESULTS_DIR.parent.parent)
    print(f"  {rel}")

## Run history

Lists all past runs and shows how aggregate metrics have changed over time.

In [ ]:
result_files = sorted(run_eval.RESULTS_DIR.glob("results_????????_??????.json"))
print(f"{len(result_files)} run(s) found\n")

header = f"  {'run_ts':17}  {'n':>4}  {'tool_sel':>8}  {'no_spur':>7}  {'no_miss':>7}  {'arg_acc':>7}  {'grounded':>8}  {'avg_llm':>7}"
print(header)
print("  " + "-" * (len(header) - 2))
for f in result_files:
    p = json.loads(f.read_text())
    a = p["aggregate"]
    print(
        f"  {p['run_ts']:17}  {a['n_cases']:4d}"
        f"  {a['tool_selection_accuracy']:>8.0%}"
        f"  {a['no_spurious_calls_rate']:>7.0%}"
        f"  {a['no_missing_calls_rate']:>7.0%}"
        f"  {a['argument_accuracy_mean'] or 0:>7.0%}"
        f"  {a['answer_groundedness_rate'] or 0:>8.0%}"
        f"  {a['avg_llm_calls']:>7.1f}"
    )